In [ ]:
# Data Preparation: Synthetic Phantom & Clinical LoDoPaB-CT Exploration

import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skimage.data import shepp_logan_phantom
from skimage.transform import radon, resize

# 1. Determine paths relative to project root
current_dir = Path('.').resolve()
root_dir = current_dir.parent if current_dir.name == 'notebooks' else current_dir
data_dir = root_dir / 'data'
real_dir = data_dir / 'real_ct_slices'
real_dir.mkdir(parents=True, exist_ok=True)

# 2. Try loading LoDoPaB dataset if available, otherwise fallback to Shepp-Logan
loaded_real = False
try:
    from dival import get_standard_dataset
    dataset = get_standard_dataset('lodopab', impl='astra_cpu')
    print(f'LoDoPaB Dataset loaded successfully! Validation samples: {len(dataset.validation)}')
    sample = dataset.validation[0]
    sinogram = sample['observation']
    ground_truth = sample['ground_truth']
    loaded_real = True
except Exception as e:
    print(f'LoDoPaB not found or not configured: {e}')
    print('Generating high-resolution synthetic Shepp-Logan phantom as demonstration...')
    ground_truth = resize(shepp_logan_phantom(), (128, 128), anti_aliasing=True)
    theta = np.linspace(0.0, 180.0, 60, endpoint=False)
    sinogram = radon(ground_truth, theta=theta, circle=True)

# 3. Visualize
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
im0 = axs[0].imshow(sinogram, cmap='gray', aspect='auto')
axs[0].set_title('Sinogram (Projections)')
plt.colorbar(im0, ax=axs[0])

im1 = axs[1].imshow(ground_truth, cmap='gray')
axs[1].set_title('Ground Truth CT Slice')
plt.colorbar(im1, ax=axs[1])

plt.tight_layout()
plt.show()

# 4. Save sample
np.save(real_dir / 'sinogram_sample0.npy', sinogram)
np.save(real_dir / 'ground_truth_sample0.npy', ground_truth)
print(f'Saved sample sinogram and ground truth to {real_dir}')
